In [3]:
!pip install pymupdf spacy scikit-learn nltk pandas ipywidgets
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 40.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
!pip install kagglehub

In [6]:
# ============================================================
# Resume Parser & Job Description Matcher
# Built from scratch - no external APIs, no HuggingFace models
# Runs on Google Colab free tier (T4 GPU / CPU)
#
# Libraries used (all open-source, no paid APIs):
#   - PyMuPDF      : PDF text extraction
#   - spaCy        : NER for resume entity parsing
#   - scikit-learn : TF-IDF vectorizer + cosine similarity for scoring
#   - nltk         : stopwords, tokenization
#   - pandas       : dataset handling
#   - ipywidgets   : Colab interactive UI
#
# Colab setup - run this cell first:
#   !pip install pymupdf spacy scikit-learn nltk pandas ipywidgets
#   !python -m spacy download en_core_web_sm
# ============================================================

import re
import os
import json
import string
import textwrap
from pathlib import Path

import fitz                          # pymupdf
import spacy
import nltk
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from IPython.display import display, clear_output
import ipywidgets as widgets

nltk.download("stopwords", quiet=True)
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    raise OSError("Run:  !python -m spacy download en_core_web_sm")


# ============================================================
# CONFIGURATION
# ============================================================

KAGGLE_TEXT_COL     = "Resume_str"      # column with resume plain text
KAGGLE_CATEGORY_COL = "Category"        # label column (set None if absent)
MAX_DATASET_SAMPLES = 20                # how many dataset resumes to score

# download dataset and resolve CSV path automatically
import kagglehub, glob
_ds_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
_csv_files = glob.glob(f"{_ds_path}/**/*.csv", recursive=True)
KAGGLE_CSV_PATH = _csv_files[0] if _csv_files else "Resume.csv"
print("Dataset path:", KAGGLE_CSV_PATH)

# ============================================================
# SKILL & KEYWORD DICTIONARIES
# Extend these to improve coverage without changing any logic.
# ============================================================

TECH_SKILLS = {
    # languages
    "python", "java", "javascript", "typescript", "c++", "c#", "golang", "go",
    "ruby", "swift", "kotlin", "scala", "r", "matlab", "bash", "shell",
    "php", "perl", "rust", "dart", "sql", "nosql",
    # frameworks / libraries
    "django", "flask", "fastapi", "spring", "react", "angular", "vue",
    "node", "nodejs", "express", "tensorflow", "pytorch", "keras", "sklearn",
    "scikit-learn", "pandas", "numpy", "spark", "hadoop", "kafka",
    "airflow", "dbt", "celery", "graphql", "rest", "grpc",
    # databases
    "postgresql", "postgres", "mysql", "sqlite", "mongodb", "redis",
    "cassandra", "dynamodb", "bigquery", "snowflake", "elasticsearch",
    # cloud / devops
    "aws", "gcp", "azure", "docker", "kubernetes", "terraform", "ansible",
    "jenkins", "github actions", "circleci", "ci/cd", "linux", "unix",
    "nginx", "apache", "microservices", "serverless", "lambda",
    # data / ml
    "machine learning", "deep learning", "nlp", "computer vision",
    "data analysis", "data engineering", "data science", "etl", "pipeline",
    "tableau", "power bi", "looker", "mlops", "llm",
    # tools
    "git", "github", "gitlab", "jira", "confluence", "figma", "postman",
}

SOFT_SKILLS = {
    "communication", "leadership", "teamwork", "collaboration", "problem solving",
    "critical thinking", "time management", "project management", "agile",
    "scrum", "kanban", "mentoring", "stakeholder management", "presentation",
    "analytical", "detail oriented", "self motivated", "adaptable",
}

EDUCATION_KEYWORDS = {
    "bachelor", "master", "phd", "doctorate", "b.s", "m.s", "b.e", "m.e",
    "mba", "b.tech", "m.tech", "degree", "university", "college", "institute",
    "computer science", "engineering", "mathematics", "statistics",
    "information technology",
}

EXPERIENCE_PATTERNS = [
    r"(\d+)\+?\s*years?\s*(?:of\s*)?experience",
    r"(\d+)\+?\s*yrs?\s*(?:of\s*)?experience",
    r"experience\s*(?:of\s*)?(\d+)\+?\s*years?",
]

EMAIL_RE = re.compile(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+")
PHONE_RE = re.compile(r"(\+?\d[\d\s\-().]{7,}\d)")
URL_RE   = re.compile(r"(https?://[^\s]+|linkedin\.com/in/[^\s]+|github\.com/[^\s]+)")

STOPWORDS = set(stopwords.words("english"))


# ============================================================
# PDF UTILITIES
# ============================================================

def extract_text_from_pdf(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text.strip()


def save_uploaded_pdf(upload_widget) -> str:
    """Save ipywidgets FileUpload content to /tmp. Returns path or None."""
    if not upload_widget.value:
        return None
    fname   = list(upload_widget.value.keys())[0]
    content = upload_widget.value[fname]["content"]
    path    = f"/tmp/{fname}"
    with open(path, "wb") as f:
        f.write(content)
    return path


# ============================================================
# TEXT UTILITIES
# ============================================================

def clean_text(text: str) -> str:
    """Lowercase, remove punctuation and stopwords."""
    text   = text.lower()
    text   = re.sub(r"[^\w\s]", " ", text)
    text   = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 1]
    return " ".join(tokens)


def extract_ngrams(text: str) -> set:
    """Return set of unigrams, bigrams, and trigrams from text."""
    tokens   = text.lower().split()
    unigrams = set(tokens)
    bigrams  = {" ".join(tokens[i:i+2]) for i in range(len(tokens) - 1)}
    trigrams = {" ".join(tokens[i:i+3]) for i in range(len(tokens) - 2)}
    return unigrams | bigrams | trigrams


# ============================================================
# RESUME ENTITY PARSER  (spaCy NER + regex + dictionary lookup)
# ============================================================

def parse_resume_entities(text: str) -> dict:
    doc = nlp(text[:50000])   # spaCy input limit safety
    return {
        "name":             _get_name(text),
        "email":            _get_email(text),
        "phone":            _get_phone(text),
        "links":            _get_links(text),
        "location":         _get_location(doc),
        "experience_years": _get_experience_years(text),
        "companies":        _get_companies(doc),
        "education":        _get_education(doc, text),
        "skills":           _get_skills(text),
    }


def _get_name(text: str) -> str:
    # look for PERSON entity in the first 6 lines (header area of a resume)
    first_block = " ".join(text.split("\n")[:6])
    for ent in nlp(first_block).ents:
        if ent.label_ == "PERSON":
            return ent.text.strip()
    return "Not found"


def _get_email(text: str) -> str:
    m = EMAIL_RE.search(text)
    return m.group(0) if m else "Not found"


def _get_phone(text: str) -> str:
    m = PHONE_RE.search(text)
    return re.sub(r"\s+", "", m.group(0)) if m else "Not found"


def _get_links(text: str) -> list:
    return URL_RE.findall(text)


def _get_location(doc) -> str:
    for ent in doc.ents:
        if ent.label_ == "GPE":        # geopolitical entity = city/state/country
            return ent.text.strip()
    return "Not found"


def _get_experience_years(text: str) -> str:
    for pat in EXPERIENCE_PATTERNS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1) + " years"
    return "Not found"


def _get_companies(doc) -> list:
    seen  = set()
    orgs  = []
    for ent in doc.ents:
        if ent.label_ == "ORG" and ent.text.lower() not in seen:
            orgs.append(ent.text.strip())
            seen.add(ent.text.lower())
    return orgs[:8]


def _get_education(doc, text: str) -> str:
    for line in text.split("\n"):
        low = line.lower()
        if any(kw in low for kw in EDUCATION_KEYWORDS) and len(line.strip()) > 5:
            return line.strip()
    # fallback: spaCy ORG containing university/college
    for ent in doc.ents:
        if ent.label_ == "ORG" and any(kw in ent.text.lower() for kw in ["university", "college", "institute"]):
            return ent.text.strip()
    return "Not found"


def _get_skills(text: str) -> list:
    ngrams = extract_ngrams(text)
    return sorted(s for s in (TECH_SKILLS | SOFT_SKILLS) if s in ngrams)


# ============================================================
# JD KEYWORD EXTRACTOR  (dictionary lookup + TF-IDF)
# ============================================================

def extract_jd_keywords(jd_text: str) -> list:
    """
    Two-pass keyword extraction:
      Pass 1 - dictionary: match known skill terms via n-gram lookup
      Pass 2 - TF-IDF: pull high-weight non-stopword tokens not in dictionary
    """
    # pass 1
    ngrams    = extract_ngrams(jd_text)
    dict_hits = {s for s in (TECH_SKILLS | SOFT_SKILLS) if s in ngrams}

    # pass 2
    sentences = sent_tokenize(jd_text)
    tfidf_hits = set()
    if len(sentences) > 1:
        vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", max_features=40)
        try:
            matrix     = vec.fit_transform(sentences)
            scores     = np.asarray(matrix.sum(axis=0)).flatten()
            names      = vec.get_feature_names_out()
            top_idx    = scores.argsort()[::-1][:20]
            tfidf_hits = {names[i] for i in top_idx}
        except ValueError:
            pass

    return sorted(dict_hits | tfidf_hits)


# ============================================================
# KEYWORD CLASSIFIER  (exact + partial matching)
# ============================================================

def classify_keywords(resume_text: str, jd_keywords: list) -> dict:
    """
    For each JD keyword classify as:
      matched  - exact n-gram found in resume
      partial  - root word, abbreviation, or partial term found
      missing  - not found at all
    """
    ngrams       = extract_ngrams(resume_text)
    resume_lower = resume_text.lower()
    matched, partial, missing = [], [], []

    for kw in jd_keywords:
        kw_lower = kw.lower()
        if kw_lower in ngrams:
            matched.append(kw)
        elif _is_partial_match(kw_lower, resume_lower, ngrams):
            partial.append(kw)
        else:
            missing.append(kw)

    return {"matched": matched, "partial": partial, "missing": missing}


def _is_partial_match(keyword: str, resume_lower: str, ngrams: set) -> bool:
    words = keyword.split()
    # any individual word of a multi-word keyword appears (length > 3 avoids noise)
    if len(words) > 1 and any(w in ngrams for w in words if len(w) > 3):
        return True
    # stem check: strip common suffixes
    for suffix in ("ing", "tion", "er", "ed", "s"):
        if keyword.endswith(suffix):
            root = keyword[:-len(suffix)]
            if len(root) > 3 and root in resume_lower:
                return True
    # abbreviation check: "machine learning" -> "ml"
    if len(words) > 1:
        abbr = "".join(w[0] for w in words)
        if abbr in ngrams:
            return True
    return False


# ============================================================
# SCORING ENGINE  (TF-IDF cosine + keyword coverage)
# ============================================================

def compute_match_score(resume_text: str, jd_text: str, classification: dict) -> int:
    """
    Score = 60% cosine similarity (TF-IDF on full texts)
           + 40% keyword coverage  (matched + 0.5 * partial) / total
    Returns integer 0-100.
    """
    # cosine similarity
    cleaned = [clean_text(resume_text), clean_text(jd_text)]
    cosine  = 0.0
    if all(cleaned):
        vec = TfidfVectorizer(ngram_range=(1, 2))
        try:
            mat    = vec.fit_transform(cleaned)
            cosine = float(cosine_similarity(mat[0], mat[1])[0][0])
        except ValueError:
            pass

    # keyword coverage
    total   = sum(len(classification[k]) for k in ("matched", "partial", "missing"))
    covered = len(classification["matched"]) + 0.5 * len(classification["partial"])
    kw_cov  = (covered / total) if total > 0 else 0.0

    return int(round((0.60 * cosine + 0.40 * kw_cov) * 100))


# ============================================================
# RECOMMENDATIONS ENGINE  (rule-based gap analysis)
# ============================================================

def generate_recommendations(
    resume_text: str,
    jd_text: str,
    classification: dict,
    entities: dict,
    score: int
) -> list:
    """
    Rule-based recommendations derived from keyword gaps, score level,
    entity presence, and quantification signals in the resume.
    """
    recs         = []
    missing      = classification.get("missing", [])
    partial      = classification.get("partial", [])
    resume_lower = resume_text.lower()

    # missing technical skills
    tech_missing = [k for k in missing if k in TECH_SKILLS][:5]
    if tech_missing:
        recs.append(
            f"Add these missing technical skills explicitly to your skills section: "
            f"{', '.join(tech_missing)}. If you have indirect experience, add a "
            f"project or bullet point that demonstrates usage."
        )

    # partial matches - exact phrasing
    tech_partial = [k for k in partial if k in TECH_SKILLS][:4]
    if tech_partial:
        recs.append(
            f"You have partial matches for: {', '.join(tech_partial)}. "
            f"Use the exact terms from the job description. ATS systems match on exact "
            f"keywords, so 'Postgres' and 'PostgreSQL' may score differently."
        )

    # missing soft skills
    soft_missing = [k for k in missing if k in SOFT_SKILLS][:3]
    if soft_missing:
        recs.append(
            f"The JD emphasizes: {', '.join(soft_missing)}. Add a brief summary or "
            f"bullet that demonstrates these with a concrete example, such as "
            f"'Led cross-functional team of 6 to deliver project on time'."
        )

    # experience years gap
    exp_str     = entities.get("experience_years", "Not found")
    exp_match   = re.search(r"(\d+)", exp_str) if exp_str != "Not found" else None
    jd_exp      = re.search(r"(\d+)\+?\s*years?", jd_text, re.IGNORECASE)
    if jd_exp and exp_match:
        jd_req, resume_exp = int(jd_exp.group(1)), int(exp_match.group(1))
        if resume_exp < jd_req:
            recs.append(
                f"The JD requires {jd_req}+ years of experience; your resume shows "
                f"{resume_exp} years. Include all internships, freelance work, and "
                f"relevant side projects to maximize total stated experience."
            )
    elif jd_exp and exp_str == "Not found":
        recs.append(
            f"The JD requires {jd_exp.group(1)}+ years of experience but your resume "
            f"does not state this explicitly. Add a professional summary line such as "
            f"'X years of experience in ...' near the top."
        )

    # quantified achievements check
    if not re.search(r"\d+%|\d+x|\$\d+|\d+\s*(users|clients|projects|team|engineers)", resume_lower):
        recs.append(
            "Your resume lacks quantified achievements. Add metrics to bullet points, "
            "e.g. 'Reduced load time by 40%', 'Managed team of 8', 'Scaled to 50K users'. "
            "Quantification significantly improves recruiter and ATS scoring."
        )

    # education
    edu_missing = [k for k in missing if k in EDUCATION_KEYWORDS]
    if edu_missing:
        recs.append(
            f"The JD mentions education requirements ({', '.join(edu_missing[:3])}). "
            f"Ensure your education section clearly states degree, field, and institution."
        )

    # low overall score
    if score < 40:
        recs.append(
            "Overall match is low. Tailor this resume specifically for this role: "
            "rewrite your professional summary to mirror JD language, reorder bullet "
            "points so the most relevant experience appears first."
        )
    elif score < 60:
        recs.append(
            "Match score is moderate. Mirror the exact phrasing used in the JD in your "
            "bullet points. ATS parsers weight exact phrase matches higher than paraphrases."
        )

    if not recs:
        recs.append(
            "Your resume is a strong match. Ensure formatting is ATS-friendly: no tables, "
            "no text boxes, standard section headings, and saved as a plain PDF."
        )

    return recs


# ============================================================
# KAGGLE DATASET LOADER
# ============================================================

def load_kaggle_resumes(csv_path: str, text_col: str, cat_col: str, n: int) -> list:
    if not Path(csv_path).exists():
        raise FileNotFoundError(f"CSV not found at '{csv_path}'. Check KAGGLE_CSV_PATH.")
    df = pd.read_csv(csv_path).dropna(subset=[text_col]).head(n)
    resumes = []
    for i, row in df.iterrows():
        resumes.append({
            "id":    f"ds_{i}",
            "text":  str(row[text_col]),
            "label": str(row[cat_col]) if cat_col and cat_col in df.columns else f"Resume #{i}",
        })
    return resumes


# ============================================================
# REPORT PRINTER
# ============================================================

W = 72  # console width

def sep(c="─"): print(c * W)

def header(title: str):
    sep("═"); print(f"  {title}"); sep("═")

def score_bar(score: int) -> str:
    filled = int(score / 5)
    return "[" + "#" * filled + "." * (20 - filled) + f"]  {score}/100"

def print_entity_table(entities: dict):
    list_fields = {"skills", "companies", "links"}
    for key, val in entities.items():
        label = key.replace("_", " ").capitalize()
        if key in list_fields:
            items = val if val else ["None"]
            print(f"  {label:<22}: {items[0]}")
            for item in items[1:]:
                print(f"  {'':<22}  {item}")
        else:
            print(f"  {label:<22}: {val}")

def print_keyword_block(label: str, items: list, marker: str):
    if not items:
        print(f"  {label}: None"); return
    print(f"  {label} ({len(items)}):")
    line = ""
    for kw in items:
        chunk = f"{marker} {kw}   "
        if len(line) + len(chunk) > W - 4:
            print("    " + line); line = chunk
        else:
            line += chunk
    if line: print("    " + line)

def print_ranked_table(results: list, user_label: str = "Your Resume"):
    col = W - 32
    print(f"  {'Rank':<5} {'Label':<{col}} {'Score':>6}  Bar")
    sep()
    for rank, r in enumerate(results, 1):
        label  = r["label"][:col]
        marker = "  <-- YOUR RESUME" if r["label"] == user_label else ""
        print(f"  {rank:<5} {label:<{col}} {r['score']:>5}%  {score_bar(r['score'])}{marker}")

def print_recommendations(recs: list):
    for i, rec in enumerate(recs, 1):
        wrapped = textwrap.fill(rec, width=W - 6, subsequent_indent="      ")
        print(f"  {i}. {wrapped}")


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline(
    job_description: str,
    resume_pdf_path: str = None,
    resume_text: str = None,
    use_kaggle: bool = True,
) -> dict:
    """
    Steps:
      1. Extract JD keywords (dictionary + TF-IDF)
      2. Parse resume entities (spaCy NER + regex + dictionary)
      3. Classify JD keywords against resume (exact + partial)
      4. Compute match score (cosine similarity + keyword coverage)
      5. Repeat steps 3-4 for each dataset resume
      6. Rank all resumes by score
      7. Generate rule-based recommendations for uploaded resume
    """
    if not job_description.strip():
        print("Error: Job description is empty."); return {}
    if not resume_pdf_path and not resume_text:
        print("Error: Provide resume_pdf_path or resume_text."); return {}

    # step 1
    header("Step 1  |  Job Description Keyword Extraction")
    jd_keywords = extract_jd_keywords(job_description)
    print(f"  Extracted {len(jd_keywords)} keywords:")
    line = ""
    for kw in jd_keywords:
        chunk = f"  {kw}   "
        if len(line) + len(chunk) > W:
            print("    " + line); line = chunk
        else:
            line += chunk
    if line: print("    " + line)

    # step 2
    header("Step 2  |  Resume Entity Extraction")
    user_text = extract_text_from_pdf(resume_pdf_path) if resume_pdf_path else resume_text
    entities  = parse_resume_entities(user_text)
    print_entity_table(entities)

    # step 3-4
    header("Step 3  |  Resume vs Job Description Scoring")
    user_class = classify_keywords(user_text, jd_keywords)
    user_score = compute_match_score(user_text, job_description, user_class)
    print(f"  Score : {score_bar(user_score)}\n")
    print_keyword_block("Matched", user_class["matched"], "+")
    print()
    print_keyword_block("Partial", user_class["partial"], "~")
    print()
    print_keyword_block("Missing", user_class["missing"], "x")

    all_results = [{"label": "Your Resume", "score": user_score, **user_class}]

    # step 5 - dataset
    if use_kaggle:
        header("Step 4  |  Dataset Resume Scoring")
        try:
            dataset = load_kaggle_resumes(KAGGLE_CSV_PATH, KAGGLE_TEXT_COL, KAGGLE_CATEGORY_COL, MAX_DATASET_SAMPLES)
            print(f"  Loaded {len(dataset)} resumes. Scoring...\n")
            sep()
            for i, entry in enumerate(dataset):
                cls   = classify_keywords(entry["text"], jd_keywords)
                score = compute_match_score(entry["text"], job_description, cls)
                all_results.append({"label": entry["label"], "score": score, **cls})
                print(f"  [{i+1:>2}/{len(dataset)}] {entry['label'][:35]:<35}  {score_bar(score)}")
        except FileNotFoundError as e:
            print(f"  Dataset skipped: {e}")

    # step 6 - rank
    all_results.sort(key=lambda r: r["score"], reverse=True)
    header("Step 5  |  Ranked Results")
    print_ranked_table(all_results)

    # step 7 - recommendations
    header("Step 6  |  Recommendations to Improve Your Resume")
    recs = generate_recommendations(user_text, job_description, user_class, entities, user_score)
    print_recommendations(recs)

    sep("═"); print("  Done."); sep("═")

    return {
        "jd_keywords":     jd_keywords,
        "parsed_resume":   entities,
        "user_score":      user_score,
        "user_class":      user_class,
        "all_results":     all_results,
        "recommendations": recs,
    }


# ============================================================
# COLAB INTERACTIVE UI
# ============================================================

def launch_ui():
    """Call launch_ui() in a Colab cell to get an interactive form."""
    style = {"description_width": "150px"}

    jd_box = widgets.Textarea(
        placeholder="Paste the full job description here...",
        description="Job Description:",
        layout=widgets.Layout(width="100%", height="200px"),
        style=style
    )
    upload_pdf = widgets.FileUpload(
        accept=".pdf", multiple=False,
        description="Upload Resume PDF",
        layout=widgets.Layout(width="100%")
    )
    resume_text_box = widgets.Textarea(
        placeholder="Or paste resume text here (leave blank if uploading PDF)...",
        description="Resume Text:",
        layout=widgets.Layout(width="100%", height="160px"),
        style=style
    )
    use_dataset = widgets.Checkbox(
        value=True,
        description="Compare against Kaggle dataset",
        style=style
    )
    run_btn = widgets.Button(
        description="Run Analysis",
        button_style="primary",
        layout=widgets.Layout(width="180px", margin="12px 0")
    )
    out = widgets.Output()

    def on_run(_):
        with out:
            clear_output()
            jd       = jd_box.value.strip()
            pdf_path = save_uploaded_pdf(upload_pdf)
            rtext    = resume_text_box.value.strip() or None
            if not jd:
                print("Please enter a job description."); return
            if not pdf_path and not rtext:
                print("Please upload a PDF or paste resume text."); return
            if pdf_path:
                print(f"PDF loaded: {pdf_path}\n")
            run_pipeline(
                job_description=jd,
                resume_pdf_path=pdf_path,
                resume_text=rtext,
                use_kaggle=use_dataset.value
            )

    run_btn.on_click(on_run)
    display(
        widgets.HTML("<h3>Resume Parser & JD Matcher</h3>"),
        jd_box, upload_pdf, resume_text_box, use_dataset, run_btn, out
    )


# ============================================================
# SCRIPT ENTRY POINT  (edit variables below and run directly)
# ============================================================

if __name__ == "__main__":

    JOB_DESCRIPTION = """
    We are looking for a Senior Python Developer with 5+ years of experience.
    Required: Python, REST APIs, SQL, Docker, AWS, Git, PostgreSQL.
    Nice to have: FastAPI, Kubernetes, CI/CD, Redis, microservices architecture.
    Strong communication skills and experience in agile teams required.
    A bachelor's degree in Computer Science or equivalent is required.
    """

    RESUME_PDF  = None   # e.g. "my_resume.pdf"  -  set to None if using text below

    RESUME_TEXT = """
    Jane Doe | jane@example.com | New York, NY
    Senior Software Engineer - 6 years of experience in backend systems.
    Skills: Python, Flask, REST APIs, PostgreSQL, Git, Docker, Linux, Agile
    Experience:
      - Backend Engineer at Acme Corp (2019-2024)
        Built microservices handling 1M+ daily requests.
        Reduced API latency by 35% through Redis caching.
      - Junior Developer at StartupXYZ (2018-2019)
    Education: B.S. Computer Science, NYU 2018
    """

    run_pipeline(
        job_description=JOB_DESCRIPTION,
        resume_pdf_path=RESUME_PDF,
        resume_text=RESUME_TEXT,
        use_kaggle=True
    )

100%|██████████| 62.5M/62.5M [00:00<00:00, 152MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/snehaanbhawal/resume-dataset/versions/1/Resume/Resume.csv
════════════════════════════════════════════════════════════════════════
  Step 1  |  Job Description Keyword Extraction
════════════════════════════════════════════════════════════════════════
  Extracted 22 keywords:
      agile     agile teams     bachelor     bachelor degree   
      communication     communication skills     computer   
      computer science     degree     degree computer     developer   
      developer years     equivalent     equivalent required   
      experience     experience agile     looking     looking senior   
      microservices     python     required     rest   
════════════════════════════════════════════════════════════════════════
  Step 2  |  Resume Entity Extraction
════════════════════════════════════════════════════════════════════════
  Name                  : Linux
  Email                 : jane@example.com
  Phone                 : 2019

In [8]:
# ================================================================
# Resume Parser & Job Description Matcher
#
# Models used:
#   - yashpwr/resume-ner-bert-v2   : NER parsing of uploaded resume
#   - TechWolf/JobBERT-v2          : semantic embeddings for scoring
#
# Dataset:
#   - snehaanbhawal/resume-dataset (2,484 resumes, 24 categories)
#     downloaded automatically via kagglehub
#
# Runs on Google Colab free tier (T4 GPU / CPU)
#
# ----------------------------------------------------------------
# Colab setup — run this cell ONCE before anything else:
#
#   !pip install kagglehub transformers sentence-transformers \
#               pymupdf torch scikit-learn pandas ipywidgets
#   !python -m spacy download en_core_web_sm   # only needed for links
#
# ================================================================

import os
import re
import json
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import kagglehub
import fitz                                          # pymupdf
import numpy as np
import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

from transformers import AutoTokenizer, AutoModelForTokenClassification
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer


# ================================================================
# CONFIGURATION
# ================================================================

KAGGLE_SLUG         = "snehaanbhawal/resume-dataset"
KAGGLE_TEXT_COL     = "Resume_str"
KAGGLE_CATEGORY_COL = "Category"
MAX_DATASET_SAMPLES = 200          # set to None to use all 2484 rows
                                   # 200 encodes in ~30s on T4

NER_MODEL_NAME      = "yashpwr/resume-ner-bert-v2"
JOBBERT_MODEL_NAME  = "TechWolf/JobBERT-v2"
NER_MAX_LEN         = 512          # NER BERT token limit
JOBBERT_BATCH_SIZE  = 32           # batch size for encoding dataset

EMBEDDING_CACHE     = "jobbert_embeddings.npy"   # cached so re-runs are fast
LABELS_CACHE        = "jobbert_labels.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# ================================================================
# STEP 1 — DOWNLOAD KAGGLE DATASET
# ================================================================

def load_dataset() -> pd.DataFrame:
    """
    Download dataset via kagglehub and return cleaned DataFrame.
    kagglehub caches the download so subsequent calls are instant.
    """
    print("Downloading dataset...")
    path      = kagglehub.dataset_download(KAGGLE_SLUG)
    csv_files = list(Path(path).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV found in {path}")
    csv_path = csv_files[0]
    print(f"CSV: {csv_path}")

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[KAGGLE_TEXT_COL])
    df = df.rename(columns={
        KAGGLE_TEXT_COL:     "text",
        KAGGLE_CATEGORY_COL: "category",
    })
    if MAX_DATASET_SAMPLES:
        # sample evenly across categories so all 24 job types are represented
        df = (
            df.groupby("category", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), max(1, MAX_DATASET_SAMPLES // df["category"].nunique()))))
            .reset_index(drop=True)
        )
    print(f"Loaded {len(df)} resumes across {df['category'].nunique()} categories.")
    return df


# ================================================================
# STEP 2 — LOAD MODELS
# ================================================================

def load_ner_model():
    """Load Resume NER BERT v2 for entity extraction."""
    print(f"Loading NER model: {NER_MODEL_NAME} ...")
    tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)
    model     = AutoModelForTokenClassification.from_pretrained(NER_MODEL_NAME)
    model.to(DEVICE)
    model.eval()
    print("NER model ready.")
    return tokenizer, model


def load_jobbert():
    """Load JobBERT-v2 SentenceTransformer for semantic similarity."""
    print(f"Loading JobBERT: {JOBBERT_MODEL_NAME} ...")
    model = SentenceTransformer(JOBBERT_MODEL_NAME, device=DEVICE)
    print("JobBERT ready.")
    return model


# ================================================================
# STEP 3 — NER RESUME PARSER
# ================================================================

def run_ner(text: str, tokenizer, model) -> list:
    """
    Run Resume NER BERT v2 on text.
    Returns list of {label, text} entity dicts.
    Handles long resumes by chunking at sentence boundaries.
    """
    # chunk text into NER_MAX_LEN token windows with 50-token overlap
    sentences  = re.split(r"(?<=[.!\n])\s+", text)
    chunks     = []
    current    = ""
    for sent in sentences:
        candidate = current + " " + sent
        tokens    = tokenizer.encode(candidate, add_special_tokens=True)
        if len(tokens) > NER_MAX_LEN - 10:
            if current:
                chunks.append(current.strip())
            current = sent
        else:
            current = candidate
    if current:
        chunks.append(current.strip())

    all_entities = []
    for chunk in chunks:
        all_entities.extend(_ner_chunk(chunk, tokenizer, model))

    # deduplicate by (label, text)
    seen  = set()
    dedup = []
    for e in all_entities:
        key = (e["label"], e["text"].lower())
        if key not in seen:
            dedup.append(e)
            seen.add(key)
    return dedup


def _ner_chunk(text: str, tokenizer, model) -> list:
    """Run NER on a single chunk, return entity list."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=NER_MAX_LEN,
        padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs     = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)[0]

    tokens    = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    id2label  = model.config.id2label
    entities  = []
    current   = None

    for token, pred in zip(tokens, predictions):
        label = id2label[pred.item()]

        if token in ("[CLS]", "[SEP]", "[PAD]"):
            if current:
                entities.append(current)
                current = None
            continue

        # merge wordpiece subwords (## prefix)
        clean_token = token.replace("##", "")

        if label.startswith("B-"):
            if current:
                entities.append(current)
            current = {"label": label[2:], "text": clean_token}

        elif label.startswith("I-") and current:
            sep = "" if token.startswith("##") else " "
            current["text"] += sep + clean_token

        else:  # O label
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    return entities


def entities_to_structured(entity_list: list) -> dict:
    """
    Group raw NER entity list into structured dict by label type.
    Label set from resume-ner-bert-v2 model card (25 entity types).
    """
    grouped = {}
    for e in entity_list:
        grouped.setdefault(e["label"], []).append(e["text"])

    # map model labels to readable field names
    label_map = {
        "Name":            "name",
        "Email Address":   "email",
        "Phone Number":    "phone",
        "Location":        "location",
        "Companies worked at": "companies",
        "Designation":     "title",
        "Skills":          "skills",
        "Degree":          "education",
        "College Name":    "institution",
        "Graduation Year": "graduation_year",
        "Years of Experience": "experience_years",
        "Linkedin":        "linkedin",
        "Github":          "github",
    }

    structured = {}
    for model_label, field in label_map.items():
        vals = grouped.get(model_label, [])
        if vals:
            structured[field] = vals[0] if len(vals) == 1 else vals

    # collect remaining labels not in map
    extra_labels = set(grouped.keys()) - set(label_map.keys())
    for lbl in extra_labels:
        structured[lbl.lower().replace(" ", "_")] = grouped[lbl]

    return structured


# ================================================================
# STEP 4 — JOBBERT EMBEDDINGS
# ================================================================

def encode_texts(texts: list, jobbert) -> np.ndarray:
    """
    Encode a list of texts into JobBERT embeddings.
    Uses the custom encode function from model card to handle batching.
    Returns float32 numpy array of shape (N, 1024).
    """
    sorted_idx   = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in sorted_idx]

    all_embeddings = []
    for i in range(0, len(sorted_texts), JOBBERT_BATCH_SIZE):
        batch = sorted_texts[i : i + JOBBERT_BATCH_SIZE]
        # use the model's internal tokenize + forward as per model card
        features = jobbert.tokenize(batch)
        features = {k: v.to(DEVICE) for k, v in features.items()}
        features["text_keys"] = ["anchor"]
        with torch.no_grad():
            out = jobbert.forward(features)
        emb = out["sentence_embedding"].cpu().numpy()
        all_embeddings.append(emb)

    sorted_emb    = np.concatenate(all_embeddings, axis=0)
    original_order = np.argsort(sorted_idx)
    return sorted_emb[original_order].astype(np.float32)


def build_or_load_dataset_embeddings(df: pd.DataFrame, jobbert) -> np.ndarray:
    """
    Encode all dataset resume texts with JobBERT.
    Saves result to EMBEDDING_CACHE so re-runs skip encoding.
    Cache is invalidated if dataset size changes.
    """
    if Path(EMBEDDING_CACHE).exists() and Path(LABELS_CACHE).exists():
        cached_labels = json.loads(Path(LABELS_CACHE).read_text())
        if len(cached_labels) == len(df):
            print(f"Loading cached embeddings ({len(df)} resumes)...")
            return np.load(EMBEDDING_CACHE)
        print("Cache size mismatch, re-encoding...")

    print(f"Encoding {len(df)} resumes with JobBERT (this runs once)...")
    texts      = df["text"].tolist()
    embeddings = encode_texts(texts, jobbert)

    np.save(EMBEDDING_CACHE, embeddings)
    Path(LABELS_CACHE).write_text(json.dumps(df["category"].tolist()))
    print("Embeddings saved to cache.")
    return embeddings


# ================================================================
# STEP 5 — SCORING
# ================================================================

def score_resume_vs_jd(resume_text: str, jd_text: str, jobbert) -> float:
    """
    Compute cosine similarity between a single resume and JD.
    Returns float 0-1.
    """
    embs = encode_texts([resume_text, jd_text], jobbert)
    sim  = float(cos_sim(embs[0:1], embs[1:2])[0][0])
    return max(0.0, min(1.0, sim))   # clamp to [0,1]


def score_dataset_vs_jd(
    df: pd.DataFrame,
    dataset_embeddings: np.ndarray,
    jd_text: str,
    jobbert
) -> pd.DataFrame:
    """
    Score all dataset resumes against the JD in one vectorized step.
    Returns df with added 'score' column (0-100 int).
    """
    jd_emb    = encode_texts([jd_text], jobbert)           # shape (1, 1024)
    sims      = cos_sim(jd_emb, dataset_embeddings)[0]     # shape (N,)
    sims_np   = sims.cpu().numpy() if hasattr(sims, "cpu") else np.array(sims)
    df        = df.copy()
    df["score"] = (np.clip(sims_np, 0, 1) * 100).astype(int)
    return df.sort_values("score", ascending=False).reset_index(drop=True)


# ================================================================
# STEP 6 — KEYWORD GAP ANALYSIS  (TF-IDF, no external model)
# ================================================================

TECH_SKILLS = {
    "python","java","javascript","typescript","c++","c#","golang","ruby","swift",
    "kotlin","scala","sql","nosql","bash","php","rust","dart","r","matlab",
    "django","flask","fastapi","spring","react","angular","vue","nodejs","express",
    "tensorflow","pytorch","keras","sklearn","pandas","numpy","spark","hadoop",
    "kafka","airflow","dbt","graphql","rest","grpc","postgresql","postgres",
    "mysql","sqlite","mongodb","redis","cassandra","dynamodb","bigquery",
    "snowflake","elasticsearch","aws","gcp","azure","docker","kubernetes",
    "terraform","ansible","jenkins","ci/cd","linux","nginx","microservices",
    "machine learning","deep learning","nlp","computer vision","data science",
    "data engineering","etl","tableau","power bi","mlops","git","github","gitlab",
    "jira","figma","agile","scrum",
}

STOPWORDS = {
    "the","and","for","are","with","this","that","have","from","they",
    "will","been","were","their","what","when","your","more","also",
    "into","than","then","these","some","such","only","other","which",
}

def extract_ngrams(text: str) -> set:
    tokens   = text.lower().split()
    unigrams = set(tokens)
    bigrams  = {" ".join(tokens[i:i+2]) for i in range(len(tokens)-1)}
    return unigrams | bigrams

def extract_jd_keywords(jd_text: str) -> list:
    ngrams    = extract_ngrams(jd_text)
    dict_hits = {s for s in TECH_SKILLS if s in ngrams}

    # TF-IDF on JD sentences for domain terms not in dictionary
    sentences = re.split(r"[.\n]", jd_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    tfidf_hits = set()
    if len(sentences) > 1:
        vec = TfidfVectorizer(ngram_range=(1,2), stop_words="english", max_features=30)
        try:
            mat    = vec.fit_transform(sentences)
            scores = np.asarray(mat.sum(axis=0)).flatten()
            names  = vec.get_feature_names_out()
            top    = scores.argsort()[::-1][:15]
            tfidf_hits = {names[i] for i in top if names[i] not in STOPWORDS}
        except ValueError:
            pass

    return sorted(dict_hits | tfidf_hits)

def classify_keywords(resume_text: str, jd_keywords: list) -> dict:
    ngrams       = extract_ngrams(resume_text)
    resume_lower = resume_text.lower()
    matched, partial, missing = [], [], []
    for kw in jd_keywords:
        kw_lower = kw.lower()
        if kw_lower in ngrams:
            matched.append(kw)
        elif any(w in ngrams for w in kw_lower.split() if len(w) > 3):
            partial.append(kw)
        else:
            missing.append(kw)
    return {"matched": matched, "partial": partial, "missing": missing}


# ================================================================
# STEP 7 — RECOMMENDATIONS  (rule-based)
# ================================================================

def generate_recommendations(
    resume_text: str,
    jd_text: str,
    classification: dict,
    structured_entities: dict,
    score: int
) -> list:
    recs         = []
    missing      = classification.get("missing", [])
    partial      = classification.get("partial", [])
    resume_lower = resume_text.lower()

    # missing technical skills
    tech_missing = [k for k in missing if k in TECH_SKILLS][:5]
    if tech_missing:
        recs.append(
            f"Add these missing technical skills to your resume: "
            f"{', '.join(tech_missing)}. Use the exact terms as they appear "
            f"in the JD — ATS systems match on exact strings."
        )

    # partial matches
    tech_partial = [k for k in partial if k in TECH_SKILLS][:4]
    if tech_partial:
        recs.append(
            f"Strengthen these partial matches: {', '.join(tech_partial)}. "
            f"Your resume contains related terms but not the exact keyword. "
            f"Use the precise phrasing from the job description."
        )

    # experience years
    jd_exp = re.search(r"(\d+)\+?\s*years?", jd_text, re.IGNORECASE)
    if jd_exp:
        jd_req   = int(jd_exp.group(1))
        exp_field = structured_entities.get("experience_years", "")
        res_exp  = re.search(r"(\d+)", str(exp_field))
        if res_exp and int(res_exp.group(1)) < jd_req:
            recs.append(
                f"JD requires {jd_req}+ years; your resume shows {res_exp.group(1)}. "
                f"Include internships, freelance, and project work to maximise stated years."
            )
        elif not res_exp:
            recs.append(
                f"JD requires {jd_req}+ years but your resume does not state experience "
                f"duration explicitly. Add a summary line: 'X years of experience in ...'."
            )

    # quantification
    if not re.search(r"\d+%|\d+x|\$\d+|\d+\s*(users|clients|team|engineers|projects)", resume_lower):
        recs.append(
            "Add quantified achievements: e.g. 'Reduced latency by 40%', "
            "'Managed team of 8', 'Scaled to 50K daily users'. "
            "Numbers make bullet points significantly stronger."
        )

    # score-level advice
    if score < 40:
        recs.append(
            "Semantic match is low. Rewrite your professional summary to directly "
            "mirror the language of the job description. Prioritise the most "
            "relevant experience at the top of each section."
        )
    elif score < 60:
        recs.append(
            "Semantic match is moderate. Align your bullet point verbs and noun "
            "phrases more closely with the JD. Example: if the JD says "
            "'designed scalable systems', use that phrase rather than 'built systems'."
        )

    if not recs:
        recs.append(
            "Strong match. Ensure the resume is ATS-friendly: no tables, no text "
            "boxes, standard section headings (Experience, Education, Skills)."
        )

    return recs


# ================================================================
# PDF UTILITY
# ================================================================

def extract_text_from_pdf(pdf_path: str) -> str:
    doc  = fitz.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text.strip()

def save_uploaded_pdf(upload_widget) -> str:
    if not upload_widget.value:
        return None
    fname   = list(upload_widget.value.keys())[0]
    content = upload_widget.value[fname]["content"]
    path    = f"/tmp/{fname}"
    with open(path, "wb") as f:
        f.write(content)
    return path


# ================================================================
# REPORT PRINTER
# ================================================================

W = 74

def sep(c="─"): print(c * W)
def header(t):  sep("═"); print(f"  {t}"); sep("═")

def score_bar(score: int) -> str:
    filled = int(score / 5)
    return "[" + "#" * filled + "." * (20 - filled) + f"]  {score}/100"

def print_entities(structured: dict):
    list_fields = {"skills", "companies"}
    for key, val in structured.items():
        label = key.replace("_", " ").capitalize()
        if isinstance(val, list):
            items = val if val else ["None"]
            print(f"  {label:<22}: {items[0]}")
            for item in items[1:6]:          # cap long skill lists at 6
                print(f"  {'':<22}  {item}")
        else:
            print(f"  {label:<22}: {val}")

def print_kw_block(label: str, items: list, marker: str):
    if not items:
        print(f"  {label}: None"); return
    print(f"  {label} ({len(items)}):")
    line = ""
    for kw in items:
        chunk = f"{marker}{kw}   "
        if len(line) + len(chunk) > W - 4:
            print("    " + line); line = chunk
        else:
            line += chunk
    if line: print("    " + line)

def print_ranked_table(df_results: pd.DataFrame, user_score: int, user_label: str = "YOUR RESUME"):
    col = W - 34
    sep()
    print(f"  {'Rank':<5} {'Category / Label':<{col}} {'Score':>6}   Bar")
    sep()
    # insert user resume row at its sorted position
    rows = [{"category": user_label, "score": user_score, "_user": True}]
    for _, row in df_results.iterrows():
        rows.append({"category": row["category"], "score": row["score"], "_user": False})
    rows.sort(key=lambda r: r["score"], reverse=True)
    for rank, row in enumerate(rows, 1):
        marker = "  <-- YOUR RESUME" if row["_user"] else ""
        label  = str(row["category"])[:col]
        print(f"  {rank:<5} {label:<{col}} {row['score']:>5}%  {score_bar(row['score'])}{marker}")

def print_recs(recs: list):
    for i, rec in enumerate(recs, 1):
        wrapped = textwrap.fill(rec, width=W-6, subsequent_indent="      ")
        print(f"  {i}. {wrapped}")


# ================================================================
# MAIN PIPELINE
# ================================================================

# module-level model/data holders so Colab doesn't reload on each run
_ner_tokenizer = None
_ner_model     = None
_jobbert       = None
_dataset_df    = None
_dataset_embs  = None


def initialize(force_reload: bool = False):
    """
    Load dataset and models into module-level variables.
    Call once. Safe to call again — skips if already loaded.
    """
    global _ner_tokenizer, _ner_model, _jobbert, _dataset_df, _dataset_embs

    if not force_reload and _jobbert is not None:
        print("Models already loaded. Call initialize(force_reload=True) to reload.")
        return

    _dataset_df              = load_dataset()
    _ner_tokenizer, _ner_model = load_ner_model()
    _jobbert                 = load_jobbert()
    _dataset_embs            = build_or_load_dataset_embeddings(_dataset_df, _jobbert)
    print("\nAll models and data ready. Call run_pipeline(...) to analyse a resume.")


def run_pipeline(
    job_description: str,
    resume_pdf_path: str = None,
    resume_text: str = None,
) -> dict:
    """
    Full pipeline:
      1. Extract keywords from JD
      2. Parse uploaded resume with NER BERT
      3. Encode resume + JD with JobBERT, compute cosine similarity score
      4. Score all dataset resumes vs JD (vectorized)
      5. Rank everything
      6. Keyword gap analysis
      7. Generate recommendations
    """
    global _ner_tokenizer, _ner_model, _jobbert, _dataset_df, _dataset_embs

    if _jobbert is None:
        print("Models not loaded. Run initialize() first.")
        return {}
    if not job_description.strip():
        print("Error: Job description is empty."); return {}
    if not resume_pdf_path and not resume_text:
        print("Error: Provide resume_pdf_path or resume_text."); return {}

    # step 1 — JD keywords
    header("Step 1  |  Job Description Keyword Extraction")
    jd_keywords = extract_jd_keywords(job_description)
    print(f"  {len(jd_keywords)} keywords found:")
    line = ""
    for kw in jd_keywords:
        chunk = f"  {kw}   "
        if len(line) + len(chunk) > W:
            print("    " + line); line = chunk
        else:
            line += chunk
    if line: print("    " + line)

    # step 2 — NER parse uploaded resume
    header("Step 2  |  Resume Entity Extraction  (NER BERT v2)")
    user_text   = extract_text_from_pdf(resume_pdf_path) if resume_pdf_path else resume_text
    raw_ents    = run_ner(user_text, _ner_tokenizer, _ner_model)
    structured  = entities_to_structured(raw_ents)
    print_entities(structured)

    # step 3 — score uploaded resume
    header("Step 3  |  Resume vs JD Semantic Score  (JobBERT-v2)")
    user_score_f = score_resume_vs_jd(user_text, job_description, _jobbert)
    user_score   = int(round(user_score_f * 100))
    print(f"  Semantic match score: {score_bar(user_score)}")

    # step 4 — score dataset resumes
    header("Step 4  |  Dataset Resume Scoring  (2484 resumes, vectorized)")
    df_scored = score_dataset_vs_jd(_dataset_df, _dataset_embs, job_description, _jobbert)
    print(f"  Top 5 from dataset:")
    sep()
    for _, row in df_scored.head(5).iterrows():
        print(f"  {row['category']:<35}  {score_bar(row['score'])}")

    # step 5 — ranked table
    header("Step 5  |  Ranked Results (Your Resume vs Dataset)")
    print_ranked_table(df_scored, user_score)

    # step 6 — keyword gap
    header("Step 6  |  Keyword Gap Analysis")
    classification = classify_keywords(user_text, jd_keywords)
    print_kw_block("Matched", classification["matched"], "+ ")
    print()
    print_kw_block("Partial", classification["partial"], "~ ")
    print()
    print_kw_block("Missing", classification["missing"], "x ")

    # step 7 — recommendations
    header("Step 7  |  Recommendations")
    recs = generate_recommendations(
        user_text, job_description, classification, structured, user_score
    )
    print_recs(recs)

    sep("═"); print("  Done."); sep("═")

    return {
        "jd_keywords":     jd_keywords,
        "parsed_resume":   structured,
        "user_score":      user_score,
        "dataset_scores":  df_scored[["category", "score"]].to_dict("records"),
        "classification":  classification,
        "recommendations": recs,
    }


# ================================================================
# COLAB INTERACTIVE UI
# ================================================================

def launch_ui():
    """Call launch_ui() in a Colab cell. Models must be initialized first."""
    style = {"description_width": "140px"}

    jd_box = widgets.Textarea(
        placeholder="Paste the full job description here...",
        description="Job Description:",
        layout=widgets.Layout(width="100%", height="200px"),
        style=style
    )
    upload_pdf = widgets.FileUpload(
        accept=".pdf", multiple=False,
        description="Upload Resume PDF",
        layout=widgets.Layout(width="100%")
    )
    resume_text_box = widgets.Textarea(
        placeholder="Or paste resume text here (leave blank if uploading PDF)...",
        description="Resume Text:",
        layout=widgets.Layout(width="100%", height="150px"),
        style=style
    )
    run_btn = widgets.Button(
        description="Run Analysis",
        button_style="primary",
        layout=widgets.Layout(width="180px", margin="12px 0")
    )
    out = widgets.Output()

    def on_run(_):
        with out:
            clear_output()
            jd       = jd_box.value.strip()
            pdf_path = save_uploaded_pdf(upload_pdf)
            rtext    = resume_text_box.value.strip() or None
            if not jd:
                print("Please enter a job description."); return
            if not pdf_path and not rtext:
                print("Please upload a PDF or paste resume text."); return
            if pdf_path:
                print(f"PDF loaded: {pdf_path}\n")
            run_pipeline(
                job_description=jd,
                resume_pdf_path=pdf_path,
                resume_text=rtext,
            )

    run_btn.on_click(on_run)
    display(
        widgets.HTML("<h3>Resume Parser &amp; JD Matcher</h3>"),
        jd_box, upload_pdf, resume_text_box, run_btn, out
    )


# ================================================================
# SCRIPT ENTRY POINT  (edit below and run directly if not using UI)
# ================================================================

if __name__ == "__main__":

    JOB_DESCRIPTION = """
    We are looking for a Senior Python Developer with 5+ years of experience.
    Required skills: Python, REST APIs, SQL, Docker, AWS, Git, PostgreSQL.
    Nice to have: FastAPI, Kubernetes, CI/CD pipelines, Redis, microservices.
    Strong communication skills and agile team experience required.
    Bachelor's degree in Computer Science or equivalent.
    """

    RESUME_PDF  = None      # set to "path/to/resume.pdf" if using a PDF file

    RESUME_TEXT = """
    Jane Doe | jane@example.com | New York, NY
    Senior Software Engineer with 6 years of backend experience.
    Skills: Python, Flask, REST APIs, PostgreSQL, Git, Docker, Linux, Agile, Redis
    Experience:
      Backend Engineer, Acme Corp (2019-2024)
        - Built microservices handling 1M+ daily requests
        - Reduced API latency by 35% using Redis caching
        - Led team of 5 engineers, delivered 3 major releases on schedule
      Junior Developer, StartupXYZ (2018-2019)
    Education: B.S. Computer Science, NYU 2018
    """

    initialize()

    run_pipeline(
        job_description=JOB_DESCRIPTION,
        resume_pdf_path=RESUME_PDF,
        resume_text=RESUME_TEXT,
    )

Device: cpu
Using Colab cache for faster access to the 'resume-dataset' dataset.
CSV: /kaggle/input/resume-dataset/Resume/Resume.csv
Loaded 192 resumes across 24 categories.
Loading NER model: yashpwr/resume-ner-bert-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

NER model ready.
Loading JobBERT: TechWolf/JobBERT-v2 ...


modules.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

2_Asym/140216480444672_Dense/model.safet(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

2_Asym/140216480445248_Dense/model.safet(…):   0%|          | 0.00/3.15M [00:00<?, ?B/s]

JobBERT ready.
Encoding 192 resumes with JobBERT (this runs once)...
Embeddings saved to cache.

All models and data ready. Call run_pipeline(...) to analyse a resume.
══════════════════════════════════════════════════════════════════════════
  Step 1  |  Job Description Keyword Extraction
══════════════════════════════════════════════════════════════════════════
  18 keywords found:
      agile     bachelor     cd     cd pipelines     ci/cd   
      computer science     degree     degree computer     developer   
      developer years     equivalent     experience     fastapi   
      fastapi kubernetes     python     required     rest     skills   
══════════════════════════════════════════════════════════════════════════
  Step 2  |  Resume Entity Extraction  (NER BERT v2)
══════════════════════════════════════════════════════════════════════════
  Email                 : jane @ example . com
  Skills                : Skills : Python , Flask , REST APIs , PostgreSQL , Git , Docker ,